In [1]:
import json
with open(//685/Australian_data/code/dpo_train_data.json', 'r', encoding='utf-8') as f:
    data1 = json.load(f)

len(data1)

3836

In [5]:
import json
import random

file1 = //685/Australian_data/code/dpo_with_correct_context.json'
file2 = //685/Australian_data/code/dpo_with_wrong_context.json'
output_file = 'dpo_train_data.json'

# Load both JSON files
with open(file1, 'r', encoding='utf-8') as f:
    data1 = json.load(f)

with open(file2, 'r', encoding='utf-8') as f:
    data2 = json.load(f)

# Merge
merged_data = data1 + data2

# Shuffle (in-place)
random.shuffle(merged_data)

# Save
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(merged_data, f, ensure_ascii=False, indent=2)


In [4]:
import json
import re
import random

jsonl_file = //685/Australian_data/code/aus_qa_train.jsonl'
output_data = []

# ----------------------------
# Load all entries first
# ----------------------------
entries = []
with open(jsonl_file, 'r', encoding='utf-8') as f:
    for line in f:
        entry = json.loads(line)

        prompt_field = entry.get('prompt', '')
        snippet_match = re.search(
            r'<snippet>\s*(.*?)\s*</snippet>',
            prompt_field,
            re.DOTALL
        )
        snippet_text = snippet_match.group(1).strip() if snippet_match else ''

        entries.append({
            "question": entry.get('question', ''),
            "answer": entry.get('answer', ''),
            "snippet": snippet_text
        })

# ----------------------------
# Build negative (wrong-context) DPO samples
# ----------------------------
for entry in entries:
    question_text = entry['question']
    correct_answer = entry['answer']
    correct_snippet = entry['snippet']

    # choose a random WRONG snippet
    other_snippets = [
        e['snippet'] for e in entries
        if e['snippet'] and e['snippet'] != correct_snippet
    ]
    if not other_snippets:
        continue

    wrong_context = random.choice(other_snippets)

    human_prompt = (
        "You will be given a context and a question. "
        "If the question can be answered from the context, answer it based only on the context. "
        "If it cannot be answered, give the answer: "
        "'Given context is not sufficient to answer.'\n\n"
        f"Context : {wrong_context}\n\n"
        f"Question : {question_text}"
    )

    output_data.append({
        "conversations": [
            {
                "from": "human",
                "value": human_prompt
            }
        ],
        # Correct behavior with WRONG context
        "chosen": {
            "from": "gpt",
            "value": "Given context is not sufficient to answer."
        },
        # Incorrect behavior we want the model to reject
        "rejected": {
            "from": "gpt",
            "value": correct_answer
        }
    })

# ----------------------------
# Save
# ----------------------------
with open('dpo_with_wrong_context.json', 'w', encoding='utf-8') as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)


In [3]:
import json
import re

jsonl_file = //685/Australian_data/code/aus_qa_train.jsonl'
output_data = []

with open(jsonl_file, 'r', encoding='utf-8') as f:
    for line in f:
        entry = json.loads(line)

        question_text = entry.get('question', '')
        answer_text = entry.get('answer', '')

        # Extract snippet content between <snippet> and </snippet>
        prompt_field = entry.get('prompt', '')
        snippet_match = re.search(
            r'<snippet>\s*(.*?)\s*</snippet>',
            prompt_field,
            re.DOTALL
        )
        snippet_text = snippet_match.group(1).strip() if snippet_match else ''

        # Build human prompt
        human_prompt = (
            "You will be given a context and a question. "
            "If the question can be answered from the context, answer it based only on the context. "
            "If it cannot be answered, give the answer: "
            "'Given context is not sufficient to answer.'\n\n"
            f"Context : {snippet_text}\n\n"
            f"Question : {question_text}"
        )

        output_data.append({
            "conversations": [
                {
                    "from": "human",
                    "value": human_prompt
                }
            ],
            "chosen": {
                "from": "gpt",
                "value": answer_text
            },
            "rejected": {
                "from": "gpt",
                "value": "Given context is not sufficient to answer."
            }
        })

# Save as JSON array
with open('dpo_with_correct_context.json', 'w', encoding='utf-8') as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)


In [10]:
import json

# Input and output files
input_file = 'test_negative.json'
output_file = //685/Australian_data/code/test_negative_best_prompt.json'

# Load existing JSON
with open(input_file, 'r', encoding='utf-8') as f:
    data = json.load(f)

# Update the prompt for each question
for qid, entry in data.items():
    context_text = entry.get('context', '')
    question_text = entry.get('question', '')

    # New prompt format
    new_prompt = (
        "You will be given a context and a question. "
        "If the question can be answered from the context, answer it based only on the context. "
        "If it cannot be answered, give the answer: 'Given context is not sufficient to answer.'\n\n"
        f"Context : {context_text}\n\n"
        f"Question : {question_text}\n\nAnswer:"
    )

    # Update the entry
    entry['prompt'] = new_prompt

# Save the updated JSON
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=2)


In [9]:
import json
import re

jsonl_file = //685/Australian_data/code/aus_qa_test.jsonl'
output_dict = {}

with open(jsonl_file, 'r', encoding='utf-8') as f:
    for idx, line in enumerate(f, start=1):
        entry = json.loads(line)
        question_id = f"question_{idx}"
        question_text = entry.get('question', '')
        answer_text = entry.get('answer', '')

        # Extract snippet content between <snippet> and </snippet>
        prompt_field = entry.get('prompt', '')
        snippet_match = re.search(r'<snippet>\s*(.*?)\s*</snippet>', prompt_field, re.DOTALL)
        snippet_text = snippet_match.group(1).strip() if snippet_match else ''

        # Build the new prompt
        prompt_text = (
            "You will be given a context and a question. "
            "If the question can be answered from the context, answer it based only on the context. "
            "If it cannot be answered, give the answer: 'Given context is not sufficient to answer.'\n\n"
            f"Context : {snippet_text}\n\n"
            f"Question : {question_text}\n\nAnswer:"
        )

        output_dict[question_id] = {
            "question": question_text,
            "chosen_answer": answer_text,
            "rejected_anaswer":"Given context is not sufficient to answer.",
            "context": snippet_text,
            "prompt": prompt_text
        }

# Save as JSON
with open('test_positive_best_prompt.json', 'w', encoding='utf-8') as f:
    json.dump(output_dict, f, ensure_ascii=False, indent=2)


In [4]:
import json
import re

jsonl_file = //685/Australian_data/code/aus_qa_test.jsonl'
output_dict = {}

with open(jsonl_file, 'r', encoding='utf-8') as f:
    for idx, line in enumerate(f, start=1):
        entry = json.loads(line)
        question_id = f"question_{idx}"
        question_text = entry.get('question', '')
        answer_text = entry.get('answer', '')

        # Extract snippet content between <snippet> and </snippet>
        prompt_field = entry.get('prompt', '')
        snippet_match = re.search(r'<snippet>\s*(.*?)\s*</snippet>', prompt_field, re.DOTALL)
        snippet_text = snippet_match.group(1).strip() if snippet_match else ''

        # Build prompt using only snippet
        prompt = (
        "You will be given a context and a question. "
        "If the question can be answered from the context, answer it based only on the context. "
        "If it cannot be answered, give the answer: 'Given context is not sufficient to answer.'\n\n"
        f"Context : {snippet_text}\n\n"
        f"Question : {question_text}\n\nAnswer:"
    )
        output_dict[question_id] = {
            "question": question_text,
            "answer": answer_text,
            "context": snippet_text,
            "prompt": prompt
        }

# Save as JSON
with open(//685/Australian_data/code/test_positive_best_prompt.json', 'w', encoding='utf-8') as f:
    json.dump(output_dict, f, ensure_ascii=False, indent=2)


In [3]:
import json
import re
import random

jsonl_file = //685/Australian_data/code/aus_qa_train.jsonl'
output_dict = {}

# First, load all entries into a list
entries = []
with open(jsonl_file, 'r', encoding='utf-8') as f:
    for line in f:
        entry = json.loads(line)
        # Extract snippet
        prompt_field = entry.get('prompt', '')
        snippet_match = re.search(r'<snippet>\s*(.*?)\s*</snippet>', prompt_field, re.DOTALL)
        snippet_text = snippet_match.group(1).strip() if snippet_match else ''
        entries.append({
            "question": entry.get('question', ''),
            "snippet": snippet_text
        })

# Now create negative test set
for idx, entry in enumerate(entries, start=1):
    question_id = f"question_{idx}"
    question_text = entry['question']

    # Pick a random snippet that is NOT the correct context
    other_snippets = [e['snippet'] for e in entries if e['snippet'] != entry['snippet']]
    if not other_snippets:
        continue  # skip if no other snippet exists
    random_context = random.choice(other_snippets)

    # Build prompt
    prompt = (
        "You will be given a context and a question. "
        "If the question can be answered from the context, answer it based only on the context. "
        "If it cannot be answered, give the answer: 'Given context is not sufficient to answer.'\n\n"
        f"Context : {context_text}\n\n"
        f"Question : {question_text}\n\nAnswer:"
    )
    output_dict[question_id] = {
        "question": question_text,
        "answer": "Given context is not sufficient to answer",
        "context": random_context,
        "prompt": prompt_text
    }

# Save as JSON
with open('test_negative.json', 'w', encoding='utf-8') as f:
    json.dump(output_dict, f, ensure_ascii=False, indent=2)


In [6]:
import json

# Load the positive test JSON
with open('test_positive.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

word_counts = []

for entry in data.values():
    answer_text = entry.get('answer', '')
    word_counts.append(len(answer_text.split()))

if word_counts:
    min_words = min(word_counts)
    max_words = max(word_counts)
    avg_words = sum(word_counts) / len(word_counts)
    print(f"Minimum words per answer: {min_words}")
    print(f"Maximum words per answer: {max_words}")
    print(f"Average words per answer: {avg_words:.2f}")
else:
    print("No data found in the JSON.")


Minimum words per answer: 18
Maximum words per answer: 259
Average words per answer: 87.34


In [8]:
import json

# Load the JSON file
with open("//685/Australian_data/code/test_positive.json", "r") as f:
    data = json.load(f)

# Update each prompt
for key, value in data.items():
    value['prompt'] += '\nIf the question cannot be answered from the given context, then answer "Given context is not sufficient to answer".'

# Optionally, save the updated JSON
with open("//685/Australian_data/code/test_positive_better_prompt.json", "w") as f:
    json.dump(data, f, indent=2)
